<a href="https://colab.research.google.com/github/nazzhannn/fathomnet2025/blob/main/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os

# 1. This is your new Kaggle Token (No file upload needed!)
os.environ['KAGGLE_API_TOKEN'] = "KGAT_3d1e1ece332e0e876a77ab4a9f799af8"

# 2. Upgrade the Kaggle CLI so it understands the new KGAT format
!pip install --upgrade kaggle

# 3. Download the FathomNet data directly to Colab
!kaggle competitions download -c fathomnet-2025

# 4. Unzip the downloaded file quietly (-q)
!unzip -q fathomnet-2025.zip -d fathomnet_data

print("✅ Authentication successful and Data Downloaded!")

100% 1.04M/1.04M [00:00<00:00, 132MB/s]

✅ Authentication successful and Data Downloaded!


In [3]:
# 1. Install the YOLOv8 library
!pip install ultralytics

# 2. Convert the FathomNet COCO data into YOLO format
from ultralytics.data.converter import convert_coco

print("Starting conversion... This might take a few minutes!")
# Point this to the folder where the train.json file was unzipped
convert_coco(labels_dir="fathomnet_data/train.json", use_segments=False)
print("✅ Conversion Complete! YOLO folders are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting conversion... This might take a few minutes!
COCO data converted successfully.
Results saved to /content/coco_converted
✅ Conversion Complete! YOLO folders are ready.


In [4]:
from ultralytics import YOLO

# 1. Load the fastest model (YOLOv8 Nano)
model = YOLO('yolov8n.pt')

# 2. Start Training!
print("🚀 Starting Training on Colab GPU...")
results = model.train(
    data='fathomnet_data/coco_converted.yaml', # The file created in Step 2
    epochs=3,        # Just 3 epochs for our first test
    imgsz=640,       # Standard image size
    batch=16,        # Good batch size for the Colab T4 GPU
    device=0,        # Forces YOLO to use the GPU
    plots=True       # Generates result graphs automatically
)

🚀 Starting Training on Colab GPU...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=fathomnet_data/coco_converted.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap

RuntimeError: Dataset 'fathomnet_data/coco_converted.yaml' error ❌ 'fathomnet_data/coco_converted.yaml' does not exist

In [10]:
# We are telling the script to use dataset_train.json and to put the images in a new folder called "images"
!python fathomnet_data/download.py fathomnet_data/dataset_train.json fathomnet_data/images

Processing annotations: 100% 23699/23699 [22:17<00:00, 17.72it/s]
Saved dataset to /content/fathomnet_data/images


In [11]:
from ultralytics.data.converter import convert_coco

print("Starting COCO to YOLO conversion...")
# This reads the JSON and creates YOLO text files for all 8,981 images
convert_coco(labels_dir="fathomnet_data/dataset_train.json", use_segments=False)
print("✅ Conversion Complete!")

Starting COCO to YOLO conversion...
COCO data converted successfully.
Results saved to /content/coco_converted2
✅ Conversion Complete!


In [24]:
import os

def find_files(root_dir):
    img_samples = []
    lbl_samples = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith(('.png', '.jpg', '.jpeg')) and 'sample_data' not in root:
                img_samples.append(os.path.join(root, file))
            if file.endswith('.txt') and 'requirements' not in file and 'sample_data' not in root:
                lbl_samples.append(os.path.join(root, file))
            if len(img_samples) >= 5 and len(lbl_samples) >= 5:
                break
    return img_samples, lbl_samples

imgs, lbls = find_files('/content')

print("📂 --- REAL IMAGE LOCATIONS ---")
for i in imgs: print(i)

print("\n📄 --- REAL LABEL LOCATIONS ---")
for l in lbls: print(l)

if lbls:
    print("\n🧪 --- LABEL CONTENT CHECK (First Line) ---")
    with open(lbls[0], 'r') as f:
        print(f"Content of {os.path.basename(lbls[0])}: {f.readline().strip()}")

📂 --- REAL IMAGE LOCATIONS ---
/content/final_data/images/train/8603_22910.png
/content/final_data/images/train/6159_17925.png
/content/final_data/images/train/2723_9207.png
/content/final_data/images/train/4291_13363.png
/content/final_data/images/train/8034_21885.png
/content/final_data/images/train/1470_5457.png
/content/final_data/images/train/6402_18479.png
/content/final_data/images/train/3792_12039.png
/content/final_data/images/train/6576.png
/content/final_data/images/train/688_2772.png
/content/final_data/images/train/3448.png
/content/final_data/images/train/393_1520.png
/content/final_data/images/train/2933_9749.png
/content/final_data/images/train/908.png
/content/final_data/images/train/1386_5192.png
/content/final_data/images/train/5567_16531.png
/content/final_data/images/train/6074_17689.png
/content/final_data/images/train/3619.png
/content/final_data/images/train/2178_7619.png
/content/final_data/images/train/3038.png
/content/final_data/images/train/538_2170.png
/co

In [25]:
import os

# Let's check a specific pair based on your list
image_check = "7641.png"
label_check = "7641.txt"

img_path = f"/content/final_data/images/train/{image_check}"
lbl_path = f"/content/final_data/labels/train/{label_check}"

print(f"🖼️ Checking Image: {img_path}")
print(f"Exists? {'✅ YES' if os.path.exists(img_path) else '❌ NO'}")

print(f"\n📄 Checking Label: {lbl_path}")
print(f"Exists? {'✅ YES' if os.path.exists(lbl_path) else '❌ NO'}")

# List first 5 labels to see their names
print("\n📝 First 5 Label Files found:")
labels = os.listdir('/content/final_data/labels/train/')[:5]
for l in labels:
    print(f"- {l}")

🖼️ Checking Image: /content/final_data/images/train/7641.png
Exists? ✅ YES

📄 Checking Label: /content/final_data/labels/train/7641.txt
Exists? ❌ NO

📝 First 5 Label Files found:


In [30]:
import os
import json

# 1. Look at the first 5 images on your disk
img_path = '/content/final_data/images/train'
if os.path.exists(img_path):
    disk_images = sorted(os.listdir(img_path))[:10]
    print(f"📁 DISK: First 10 Images in {img_path}:")
    for img in disk_images:
        print(f"  - {img}")
else:
    print(f"❌ ERROR: {img_path} does not exist!")

# 2. Look at the first 5 labels on your disk
lbl_path = '/content/final_data/labels/train'
if os.path.exists(lbl_path):
    disk_labels = sorted(os.listdir(lbl_path))[:10]
    print(f"\n📄 DISK: First 10 Labels in {lbl_path}:")
    for lbl in disk_labels:
        print(f"  - {lbl}")
else:
    print(f"❌ ERROR: {lbl_path} does not exist!")

# 3. Look at how the database defines these files
json_path = '/content/fathomnet_data/dataset_train.json'
if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        db = json.load(f)
    print(f"\n📖 DATABASE: First 5 image entries in JSON:")
    for img_entry in db['images'][:5]:
        print(f"  - ID: {img_entry['id']} | File_Name: {img_entry['file_name']}")
else:
    print(f"❌ ERROR: {json_path} not found!")

📁 DISK: First 10 Images in /content/final_data/images/train:
  - 1.png
  - 10.png
  - 100.png
  - 1000.png
  - 1000_3865.png
  - 1000_3866.png
  - 1000_3867.png
  - 1000_3868.png
  - 1000_3869.png
  - 1000_3870.png

📄 DISK: First 10 Labels in /content/final_data/labels/train:
  - 000a4c7c-9837-4f36-b7b3-cccb85171b3d.txt
  - 0013b150-a46f-41a0-a32c-d8cbd9e6c48f.txt
  - 001faa49-b8e0-46b4-a743-9002613d69c4.txt
  - 0020ef94-baae-4358-b142-300a2dfac09f.txt
  - 00264f7f-bcda-4c38-9477-fb13caea4d98.txt
  - 0026b62d-3ebc-4a70-b62e-5767a4e5f8b8.txt
  - 00273137-b2bb-4c09-b9c5-9c3de2b0c30d.txt
  - 00316d26-a75c-41f6-a81d-4b9af7f60e60.txt
  - 004aea4b-f3b6-4afe-9728-b21b6b609f32.txt
  - 004da3fe-804f-4a6a-b8b7-50ac3998a87b.txt

📖 DATABASE: First 5 image entries in JSON:
  - ID: 1 | File_Name: 67cda248-6d28-4801-9c4e-e4525189ea38.png
  - ID: 2 | File_Name: d9b399f3-8628-4138-a339-f4520be751c5.png
  - ID: 3 | File_Name: 3289c3e1-40f2-4512-992f-3fa406b50a86.png
  - ID: 4 | File_Name: 801dc37d-1aac-

In [1]:
import os
import json

# 1. Setup paths
json_path = '/content/fathomnet_data/dataset_train.json'
label_dir = '/content/final_data/labels/train'

print("🧬 Loading translator from JSON...")
with open(json_path, 'r') as f:
    db = json.load(f)

# 2. Create a "UUID to ID" dictionary
# Maps '0026b62d...png' -> '5'
translator = {}
for img in db['images']:
    uuid_name = os.path.basename(img['file_name']).rsplit('.', 1)[0]
    translator[uuid_name] = str(img['id'])

print("📝 Renaming labels to match image IDs...")
count = 0
for label_file in os.listdir(label_dir):
    if label_file.endswith('.txt'):
        uuid_part = label_file.rsplit('.', 1)[0]

        if uuid_part in translator:
            new_name = translator[uuid_part] + ".txt"
            old_path = os.path.join(label_dir, label_file)
            new_path = os.path.join(label_dir, new_name)

            # Rename the file
            os.rename(old_path, new_path)
            count += 1

print(f"✅ SUCCESS: Renamed {count} labels!")
print(f"Sample: The label for image #1 is now named '{os.path.exists('/content/final_data/labels/train/1.txt') and '1.txt' or 'Missing'}'")

# 3. Clear the bad cache so YOLO rescans
os.system("rm -f /content/final_data/images/train.cache")

print("\n🚀 RESTARTING TRAINING... THIS IS THE ONE!")
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(
    data='/content/final_fathomnet.yaml',
    epochs=3,
    imgsz=640,
    batch=16,
    device=0
)

🧬 Loading translator from JSON...


FileNotFoundError: [Errno 2] No such file or directory: '/content/fathomnet_data/dataset_train.json'

In [33]:
# Lowering confidence to 0.01 to see the AI's "guesses"
results = model.predict(source='/content/final_data/images/train/1.png', save=True, conf=0.01)


image 1/1 /content/final_data/images/train/1.png: 352x640 1 Apostichopus leukothele, 1 Benthocodon pedunculata, 1 Heterochone calyx, 1 Hexactinellida, 1 Sebastes diploproa, 1 Sebastolobus, 1 Staurocalyptus, 7.2ms
Speed: 1.9ms preprocess, 7.2ms inference, 5.7ms postprocess per image at shape (1, 3, 352, 640)
Results saved to /content/runs/detect/predict


In [ ]:
from ultralytics import YOLO

# 1. Load the weights from your last successful run (train12)
# This ensures we don't lose the progress you've already made!
model = YOLO('/content/runs/detect/train12/weights/last.pt')

print("🚀 Starting the LONG RUN (50 Epochs)...")

# 2. Fire the engine
results = model.train(
    data='/content/final_fathomnet.yaml',
    epochs=50,       # 50 is the 'Goldilocks' zone for first-time training
    imgsz=640,       # High resolution for those tiny deep-sea creatures
    batch=16,        # Maximize the T4 GPU memory
    patience=10,     # 'Early Stopping': If it stops improving for 10 rounds, it quits to save time
    save=True,
    device=0,
    project='fathomnet_final', # Organize your output
    name='deep_sea_detect'
)

🚀 Starting the LONG RUN (50 Epochs)...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/final_fathomnet.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/train12/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=deep_sea_detect, nbs=64, nms=False, opset=No

In [ ]:
from google.colab import drive
drive.mount('/content/drive')